In [1]:
%load_ext autoreload
%autoreload 2

from comcatalyst.benchmark.analyze_results import *

# **Load results.**

In [2]:
# load from json
res = load_results_from_dir("./")

In [4]:
# restrict seeds
res = [r for r in res if r["hyperparams"]["seed"]<15]

In [5]:
# restrict number of points
res = slice_results_at_n_points(results=res, n_points=30)

In [6]:
# check
print("Total runs:", len(res))
print("n_init values:", sorted(set(r["hyperparams"]["n_init"] for r in res)))
print("n_iter_bo values:", sorted(set(r["hyperparams"]["n_iter_bo"] for r in res)))
print("sigma_noise values:", sorted(set(r["hyperparams"]["sigma_noise"] for r in res)))
print("bo_criterion values:", sorted(set(r["hyperparams"]["bo_criterion"] for r in res)))
print("kernel values:", sorted(set(r["hyperparams"]["kernel"] for r in res)))
print("seeds:", sorted(set(r["hyperparams"]["seed"] for r in res)))

Total runs: 1
n_init values: [2]
n_iter_bo values: [28]
sigma_noise values: [0.1]
bo_criterion values: ['EI']
kernel values: ['EXP_HOMO_HSPHERE']
seeds: [0]


# **Plot results.**

## **Convergence curve.**

In [ ]:
plot_convergence_curve(
    results=res,
    metric="y_true_at_x_data",
    filters={"n_init": 6, "bo_criterion": "LCB", "kernel": "CONT_RELAX", "sigma_noise": 0.1},
    figsize=(15,8)
)

## **Boxplots.**

##

### **Boxplots _y_ by n_init.**

In [ ]:
# y by n_init
for kernel in ["GOWER", "CONT_RELAX", "EXP_HOMO_HSPHERE"]:
    for bo_criterion in ["EI", "LCB"]:
        for sigma_noise in [0.1, 0.5, 1.0]:
            plot_boxplot(
                results=res,
                metric="y_true_at_x_opt",
                group_by="n_init",
                filters={
                    "kernel": kernel,
                    "bo_criterion": bo_criterion,
                    "sigma_noise": sigma_noise,
                },
                ylim = [0, 5.5]
            )

### **Boxplots _y_ by bo_criterion.**

In [ ]:
# y by bo_criterion
for kernel in ["GOWER", "CONT_RELAX", "EXP_HOMO_HSPHERE"]:
    for sigma_noise in [0.1, 0.5, 1.0]:
        plot_boxplot(
            results=res,
            metric="y_true_at_x_opt",
            group_by="bo_criterion",
            filters={
                "kernel": kernel,
                "sigma_noise": sigma_noise,
            },
            ylim=(0, 5.5)
        )

### **Boxplots _y_ by kernel.**

In [ ]:
# y by kernel
for bo_criterion in ["EI", "LCB"]:
    for sigma_noise in [0.1, 0.5, 1.0]:
        plot_boxplot(
            results=res,
            metric="y_true_at_x_opt",
            group_by="kernel",
            filters={
                "bo_criterion": bo_criterion,
                "sigma_noise": sigma_noise,
            },
            ylim=(0, 5.5)
        )

### **Boxplots _convergence_ by n_init.**

In [ ]:
for kernel in ["GOWER", "CONT_RELAX", "EXP_HOMO_HSPHERE"]:
    for bo_criterion in ["EI", "LCB"]:
        for sigma_noise in [0.1, 0.5, 1.0]:
            plot_boxplot(
                results=res,
                metric="convergence_iter_bo",
                group_by="n_init",
                filters={
                    "kernel": kernel,
                    "bo_criterion": bo_criterion,
                    "sigma_noise": sigma_noise,
                },
            )

# **Investigate results.**

In [12]:
from comcatalyst.context.cases import case_01, case_02, case_03
case = case_01()

In [10]:
summarize_results(res, epsilon=0.5)

N runs=1

y_opt mean=5.276, std=0.000, min=5.276, max=5.276
y_true_at_x_opt mean=5.266, std=0.000, min=5.266, max=5.266
conv_iter_bo mean=26.0, std=0.0, min=26, max=26

Success rate gap=0.5: 1/1=100.0%

Correlation y_opt vs conv_iter_bo: nan
Correlation y_true_at_x_opt vs conv_iter_bo: nan

Correlation y_opt vs y_init_max: nan
Correlation y_true_at_x_opt vs y_init_max: nan

Correlation y_opt vs seed: nan
Correlation y_true_at_x_opt vs seed: nan


c:\Users\siyanasran\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\siyanasran\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\siyanasran\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


In [14]:
stats_x_opt_qual_combos(
    results=res,
    vars_metadata=case.vars_metadata,
    metric="y_true_at_x_opt",
    metric_range=(4.0, 6.0),
    filters={"sigma_noise": 0.1},
)


1 runs with y_true_at_x_opt in [4.000, 6.000]

Combo                   n       %
-----------------------------------
A/alpha                 1  100.0%


In [ ]:
details_doe_by_metric_range(
    results=res,
    vars_metadata=case.vars_metadata,
    metric="y_true_at_x_opt",
    metric_range=(-10, 10),
    filters={"bo_criterion": "LCB", "kernel": "CONT_RELAX", "sigma_noise": 0.1},
    top_k=5
)

In [ ]:
plot_boxplot_by_seed(
    results = res,
    metric  = "y_true_at_x_opt",
    filters={"bo_criterion": "LCB", "kernel": "CONT_RELAX", "sigma_noise": 1.0, "n_init": 2},
)

In [ ]:
best = sorted(res, key=lambda r: r["y_true_at_x_opt"], reverse=False)[0]

plot_trajectory_by_qual(
    result = best,
    vars_metadata = case.vars_metadata,
    quant_idx_1 = 2,
    quant_idx_2 = 3,
    x_true_max = [0, 0, 0.48, 0.99],
    title = f"Best run | y_true={best['y_true_at_x_opt']:.3f}",
)